# 02 — Unified Preprocessing Pipeline
Cleans raw data and outputs ready-to-use DataFrames / CSVs for all model tiers.

Outputs:
- `data/processed/train_clean.csv`
- `data/processed/val_clean.csv`
- `data/processed/unlabeled_clean.csv`
- `data/processed/hidden_test_clean.csv`

In [1]:
import pandas as pd
import numpy as np
import re, ast, unicodedata, json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_DIR  = Path('data')
OUT_DIR  = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_ASPECTS   = ['food','service','price','cleanliness','delivery',
                   'ambiance','app_experience','general','none']
VALID_SENTIMENTS = ['positive','negative','neutral']

print('Setup complete ✓')

Setup complete ✓


In [2]:
# ── Columns to keep in each output ──────────────────────────────────────────
# Review-level (train / val / unlabeled)
REVIEW_COLS = [
    'review_id',
    'text_clean',        # cleaned Arabic text  → model input
    'text_raw',          # original text        → LLM / retrieval
    'aspects',           # list of aspects
    'aspect_sentiments', # dict  aspect → sentiment
    'star_rating_norm',  # auxiliary signal
    'is_app_platform',   # platform flag
    'char_len',
    'word_len',
    'arabic_ratio',
    'has_emoji',
]

# Flat per-aspect  (train_flat / val_flat)
FLAT_COLS = [
    'review_id',
    'text_clean',
    'text_raw',
    'aspect',
    'sentiment',
    'star_rating_norm',
    'is_app_platform',
]

print('Column lists defined ✓')


Column lists defined ✓


## 1. Load Raw Files

In [3]:
train     = pd.read_excel(RAW_DIR / r"C:\Users\HP\Downloads\DeepX_train.xlsx")
val       = pd.read_excel(RAW_DIR / r"C:\Users\HP\Downloads\DeepX_validation.xlsx")
unlabeled = pd.read_excel(RAW_DIR / r"C:\Users\HP\Downloads\DeepX_unlabeled.xlsx")

print('Shapes — train:', train.shape, '| val:', val.shape,
      '| unlabeled:', unlabeled.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\HP\\Downloads\\DeepX_train.xlsx'

## 2. Text Cleaning Functions

In [ ]:
def normalize_arabic(text):
    # ONLY normalize critical variations
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)
    # ❌ DO NOT replace ة or ى
    return text

def clean_text(text):
    if not isinstance(text, str):
        return ''

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove emojis properly
    text = re.sub(r'[^\u0600-\u06FFa-zA-Z0-9\s.,!?]', ' ', text)

    # Normalize Arabic
    text = normalize_arabic(text)

    # Remove diacritics
    text = re.sub(r'[\u064B-\u065F]', '', text)

    # Reduce repetition to ONE char (better for ML)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Clean spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text
sample = 'المكاننننن رائعةةةةةة جداااا ❤️❤️❤️ https://example.com'
print('Before:', sample) 
print('After :', clean_text(sample))

Before: المكاننننن رائعةةةةةة جداااا ❤️❤️❤️ https://example.com
After : المكان رايعة جدا


## 3. Label Parsing & Validation

In [ ]:
def safe_parse(x):
    if isinstance(x, (list, dict)):
        return x
    try:
        return ast.literal_eval(str(x))
    except Exception:
        return None

def validate_aspects(aspects):
    """Return only valid aspects."""
    if not isinstance(aspects, list):
        return []
    return [a for a in aspects if a in VALID_ASPECTS]

def validate_sentiments(sentiments, aspects):
    """Return validated sentiment dict aligned to validated aspects."""
    if not isinstance(sentiments, dict):
        return {}
    result = {}
    for a in aspects:
        s = sentiments.get(a)
        if s in VALID_SENTIMENTS:
            result[a] = s
        else:
            result[a] = 'neutral'  # fallback
    return result

def parse_and_validate_labels(df):
    df = df.copy()
    df['aspects']           = df['aspects'].apply(safe_parse)
    df['aspect_sentiments'] = df['aspect_sentiments'].apply(safe_parse)
    df['aspects']           = df['aspects'].apply(validate_aspects)
    df['aspect_sentiments'] = df.apply(
        lambda r: validate_sentiments(r['aspect_sentiments'], r['aspects']), axis=1)
    return df

train = parse_and_validate_labels(train)
val   = parse_and_validate_labels(val)
print('Labels parsed and validated ✓')

Labels parsed and validated ✓


## 4. Feature Engineering

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    text = df['review_text'].astype(str)
    
    # Cleaned text (for models)
    df['text_clean'] = text.apply(clean_text)
    
    # Original text (for LLM / retrieval)
    df['text_raw'] = text
    
    # Numeric features
    df['char_len']     = text.apply(len)
    df['word_len']     = text.apply(lambda x: len(x.split()))
    df['arabic_ratio'] = text.apply(
        lambda x: len(re.findall(r'[\u0600-\u06FF]', x)) / max(len(x), 1))
    df['has_emoji']    = text.apply(
        lambda x: int(bool(re.search(r'[\U00010000-\U0010ffff]', x))))
    df['star_rating_norm'] = pd.to_numeric(df.get('star_rating'), errors='coerce').fillna(3) / 5.0
    
    # Platform one-hot (app_store / play_store → strong app_experience signal)
    df['is_app_platform'] = df.get('platform', pd.Series([''] * len(df))).isin(
        ['play_store', 'app_store']).astype(int)
    
    return df

train     = add_features(train)
val       = add_features(val)
unlabeled = add_features(unlabeled)

print('Features added ✓')
train[['text_raw', 'text_clean', 'char_len', 'word_len', 'arabic_ratio']].head()

Features added ✓


,text_raw,text_clean,char_len,word_len,arabic_ratio
0,لا يوجد الدفع بالبطاقه عند الاستلام,لا يوجد الدفع بالبطاقه عند الاستلام,35,6,0.857143
1,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,94,16,0.755319
2,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,تجربة سيية سالتهم الاكل هياخد وقت قد ايه قالول...,104,21,0.807692
3,احلي مكان فزايد,احلي مكان فزايد,15,3,0.866667
4,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,الفطير حلو جدا الاحجام تحفة بالنسبه للسعر فا ي...,288,56,0.805556


## 5. Build Flat (per-aspect) Training Records

In [ ]:
def explode_to_flat(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates one row per (review, aspect) pair.
    Used for sequence-classification models (Tier 2).
    """
    rows = []
    for _, row in df.iterrows():
        if not isinstance(row['aspects'], list) or len(row['aspects']) == 0:
            continue
        for asp in row['aspects']:
            sent = row['aspect_sentiments'].get(asp, 'neutral') \
                   if isinstance(row['aspect_sentiments'], dict) else 'neutral'
            rows.append({
                'review_id'    : row['review_id'],
                'text_clean'   : row['text_clean'],
                'text_raw'     : row['text_raw'],
                'aspect'       : asp,
                'sentiment'    : sent,
                'star_rating_norm': row['star_rating_norm'],
                'is_app_platform' : row['is_app_platform'],
            })
    return pd.DataFrame(rows)

train_flat = explode_to_flat(train)
val_flat   = explode_to_flat(val)

print('Flat train:', train_flat.shape)
print('Flat val  :', val_flat.shape)
print('Sentiment distribution (flat train):')
print(train_flat['sentiment'].value_counts())

Flat train: (3333, 7)
Flat val  : (840, 7)
Sentiment distribution (flat train):
sentiment
positive    1646
negative    1538
neutral      149
Name: count, dtype: int64


## 5b. Handle Class Imbalance

In [ ]:
from collections import Counter

# ── 1. Diagnose ───────────────────────────────────────────────────────────────
aspect_counts   = Counter(train_flat['aspect'])
sentiment_counts = Counter(train_flat['sentiment'])

print('Aspect distribution (raw):')
for k, v in sorted(aspect_counts.items(), key=lambda x: -x[1]):
    print(f'  {k:<20} {v:>5}')

print()
print('Sentiment distribution (raw):', dict(sentiment_counts))

# ── 2. Aspect-level oversampling ──────────────────────────────────────────────
# Strategy: oversample every minority aspect up to the median count.
# We deliberately do NOT upsample to the *maximum* (service/food)
# because that would create an enormous training set; median is a
# pragmatic middle ground that balances without exploding size.

median_count = int(np.median(list(aspect_counts.values())))
MAX_OVERSAMPLE = median_count          # cap per-aspect rows at this

print(f'\nTarget count per aspect (median): {median_count}')

balanced_parts = []
for asp, grp in train_flat.groupby('aspect'):
    n_needed = max(0, MAX_OVERSAMPLE - len(grp))
    if n_needed > 0:
        # random oversampling with replacement
        oversampled = grp.sample(n=n_needed, replace=True, random_state=42)
        grp = pd.concat([grp, oversampled], ignore_index=True)
    else:
        # already at or above median — keep as-is (no under-sampling)
        pass
    balanced_parts.append(grp)

train_flat_balanced = pd.concat(balanced_parts, ignore_index=True).sample(
    frac=1, random_state=42).reset_index(drop=True)

# ── 3. Sentiment-level class weights (for loss weighting at train time) ───────
# Returned as a dict  {sentiment_label: weight}  to be passed to the model.
total = len(train_flat_balanced)
n_classes = train_flat_balanced['sentiment'].nunique()
sentiment_weights = {
    label: round(total / (n_classes * cnt), 4)
    for label, cnt in Counter(train_flat_balanced['sentiment']).items()
}

print('\nAspect distribution after balancing:')
for k, v in sorted(Counter(train_flat_balanced['aspect']).items(), key=lambda x: -x[1]):
    print(f'  {k:<20} {v:>5}')

print()
print('Sentiment class weights (for loss fn):', sentiment_weights)
print(f'\ntrain_flat rows: {len(train_flat)}  →  balanced: {len(train_flat_balanced)}')

# ── 4. Persist weights for downstream notebooks ───────────────────────────────
import json as _json
weights_path = OUT_DIR / 'sentiment_class_weights.json'
_json.dump(sentiment_weights, open(weights_path, 'w'))
print(f'Saved class weights → {weights_path}')


## 6. Save Processed Files

In [ ]:
# ── Prune to relevant columns only ───────────────────────────────────────────
def keep_cols(df, cols):
    """Keep only columns that actually exist (safe against optional fields)."""
    return df[[c for c in cols if c in df.columns]]

train_out     = keep_cols(train,     REVIEW_COLS)
val_out       = keep_cols(val,       REVIEW_COLS)
unlabeled_out = keep_cols(unlabeled, REVIEW_COLS)

train_flat_out          = keep_cols(train_flat,          FLAT_COLS)
train_flat_balanced_out = keep_cols(train_flat_balanced, FLAT_COLS)
val_flat_out            = keep_cols(val_flat,            FLAT_COLS)

# ── Save ──────────────────────────────────────────────────────────────────────
train_out.to_csv(OUT_DIR / 'train_clean.csv',                  index=False)
val_out.to_csv(OUT_DIR / 'val_clean.csv',                      index=False)
unlabeled_out.to_csv(OUT_DIR / 'unlabeled_clean.csv',          index=False)

train_flat_out.to_csv(OUT_DIR / 'train_flat.csv',              index=False)
train_flat_balanced_out.to_csv(OUT_DIR / 'train_flat_balanced.csv', index=False)
val_flat_out.to_csv(OUT_DIR / 'val_flat.csv',                  index=False)

print('Saved all processed files to', OUT_DIR)
print('Files:', [p.name for p in sorted(OUT_DIR.glob('*.csv'))])
print()
print('Column sets:')
print('  review-level  :', train_out.columns.tolist())
print('  flat          :', train_flat_out.columns.tolist())


## 7. Preprocessing Summary

In [ ]:
print('=== Preprocessing Summary ===')
print(f'Train reviews        : {len(train_out)}')
print(f'Val reviews          : {len(val_out)}')
print(f'Unlabeled reviews    : {len(unlabeled_out)}')
print()
print(f'train_flat (raw)     : {len(train_flat_out)} rows')
print(f'train_flat (balanced): {len(train_flat_balanced_out)} rows')
print(f'val_flat             : {len(val_flat_out)} rows')
print()
print('Sentiment class weights:', sentiment_weights)
print()
print('Outputs in', OUT_DIR)
for f in sorted(OUT_DIR.glob('*')):
    print(' ', f.name)
